In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
path = "/content/drive/MyDrive/tredence_ai_case_study"
os.chdir(path)

# Verify the GPU
import torch
print(f"Using GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU found'}")

Mounted at /content/drive
Using GPU: Tesla T4


In [2]:
%%writefile layers.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(PrunableLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Standard parameters
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))

        # Learnable gates (Start neutral so the optimizer discovers what to prune)
        self.gate_scores = nn.Parameter(torch.Tensor(out_features, in_features))

        self.reset_parameters()

    def reset_parameters(self):
        # Kaiming initialization for weights (Industry standard for ReLu networks)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)

        # Initialize gates at 0.0 (Sigmoid(0) = 0.5)
        nn.init.constant_(self.gate_scores, 0.0)

    def forward(self, x):
        # 1. Soft gates (differentiable)
        soft_gates = torch.sigmoid(self.gate_scores)

        # 2. Hard threshold (Physical Zeros for the forward pass)
        hard_mask = (soft_gates >= 0.01).float()

        # 3. Straight-Through Estimator (The "Senior" Math Trick)
        # Forward pass uses `hard_mask`.
        # Backward pass gradients flow through `soft_gates`.
        ste_gates = hard_mask.detach() - soft_gates.detach() + soft_gates

        # Apply the absolute mask to the weights
        pruned_weights = self.weight * ste_gates

        return F.linear(x, pruned_weights, self.bias)

Writing layers.py


In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
from layers import PrunableLinear

class CIFAR10PrunableNet(nn.Module):
    def __init__(self):
        super(CIFAR10PrunableNet, self).__init__()

        # Standard CNN backbone for high accuracy on CIFAR-10
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # The Prunable Head (where we apply the case study logic)
        # Input to linear is 128 * 4 * 4 = 2048
        self.classifier = nn.Sequential(
            PrunableLinear(2048, 512),
            nn.ReLU(),
            PrunableLinear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.classifier(x)
        return x

Overwriting models.py


In [ ]:
%%writefile data_loader.py
import torch
import torchvision
import torchvision.transforms as transforms

def get_cifar10_loaders(batch_size=128):
    """
    Downloads and prepares CIFAR-10 DataLoaders.
    Optimized for GPU with pin_memory=True.
    """
    # Augmentation for the training set to prevent overfitting during pruning
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # Standard normalization for the test set
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # Datasets
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test)

    # DataLoaders (T4 GPU Optimization: pin_memory=True, num_workers=2)
    trainloader = torch.utils.data.DataLoader(
        trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return trainloader, testloader

Overwriting data_loader.py


In [3]:
%%writefile engine.py
import torch
from layers import PrunableLinear

def get_sparsity_loss(model):
    """
    Calculates the L1 penalty of the gates using the MEAN.
    Using mean() instead of sum() makes the Lambda hyperparameter
    scale-invariant (it works the same even if we add more layers).
    """
    l1_loss = torch.tensor(0.0, device=next(model.parameters()).device)
    total_elements = 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)
            l1_loss += gates.sum()
            total_elements += gates.numel()

    # Normalize the loss
    if total_elements > 0:
        return l1_loss / total_elements
    return l1_loss

def train_one_epoch(model, dataloader, criterion, optimizer, device, lambda_val):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        # Classification Loss
        cls_loss = criterion(outputs, targets)

        # Scale-Invariant Sparsity Loss
        sparsity_loss = get_sparsity_loss(model)

        # Total Loss
        loss = cls_loss + (lambda_val * sparsity_loss)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def calculate_sparsity(model, threshold=0.01):
    total_weights = 0
    pruned_weights = 0

    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, PrunableLinear):
                gates = torch.sigmoid(module.gate_scores)
                total_weights += gates.numel()
                pruned_weights += (gates < threshold).sum().item()

    if total_weights == 0:
        return 0.0
    return (pruned_weights / total_weights) * 100.0

Writing engine.py


In [ ]:
%%writefile analysis.py
import torch
import matplotlib.pyplot as plt
import numpy as np
from layers import PrunableLinear

def plot_gate_distributions(model, lambda_val, save_path=None):
    """
    Extracts all gate scores from PrunableLinear layers, applies the sigmoid,
    and plots a histogram to visualize the sparsity distribution.
    """
    all_gates = []

    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, PrunableLinear):
                gates = torch.sigmoid(module.gate_scores).cpu().numpy().flatten()
                all_gates.extend(gates)

    if not all_gates:
        print("No PrunableLinear layers found.")
        return

    plt.figure(figsize=(8, 5))
    plt.hist(all_gates, bins=50, color='teal', alpha=0.7, edgecolor='black')
    plt.title(f'Gate Value Distribution (Lambda = {lambda_val})')
    plt.xlabel('Gate Value (Sigmoid Output)')
    plt.ylabel('Frequency (Number of Weights)')
    plt.grid(axis='y', alpha=0.75)

    # Add a vertical line to show our 0.01 pruning threshold
    plt.axvline(x=0.01, color='red', linestyle='dashed', linewidth=2, label='Pruning Threshold (0.01)')
    plt.legend()

    if save_path:
        plt.savefig(save_path)
        print(f"Plot saved to {save_path}")

    plt.show()

def print_markdown_table(results):
    """
    Generates the final comparative table required by the case study.
    """
    print("\n### Final Experiment Results")
    print("| Lambda | Test Accuracy (%) | Sparsity Level (%) |")
    print("| :--- | :--- | :--- |")
    for res in results:
        print(f"| {res['lambda']} | {res['accuracy']:.2f}% | {res['sparsity']:.2f}% |")

Overwriting analysis.py


In [8]:
%%writefile run_experiments.py
import torch
import torch.nn as nn
import torch.optim as optim
import os
import random
import numpy as np

from models import CIFAR10PrunableNet
from data_loader import get_cifar10_loaders
from engine import train_one_epoch, evaluate, calculate_sparsity
from analysis import plot_gate_distributions, print_markdown_table

# 1. Ensure Reproducibility (Crucial for Engineering)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def main():
    set_seed(42) # Lock in the random factors

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Executing on device: {device}")
    os.makedirs('outputs', exist_ok=True)

    batch_size = 128
    epochs = 20
    learning_rate = 0.001

    # Because we now use .mean() instead of .sum(), we must increase Lambda
    # 0.0: Baseline | 1.0: Moderate Sparsity | 2.5: High Sparsity
    lambda_values = [0.0, 10.0, 50.0, 100.0]

    trainloader, testloader = get_cifar10_loaders(batch_size=batch_size)
    results = []

    for lam in lambda_values:
        print(f"\n{'='*40}")
        print(f"Starting Experiment with Lambda: {lam}")
        print(f"{'='*40}")

        model = CIFAR10PrunableNet().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        for epoch in range(epochs):
            train_loss, train_acc = train_one_epoch(
                model, trainloader, criterion, optimizer, device, lam
            )
            current_sparsity = calculate_sparsity(model)

            print(f"Epoch [{epoch+1}/{epochs}] | Loss: {train_loss:.4f} | "
                  f"Train Acc: {train_acc:.2f}% | Sparsity: {current_sparsity:.2f}%")

        print("Evaluating on Test Set...")
        test_loss, test_acc = evaluate(model, testloader, criterion, device)
        final_sparsity = calculate_sparsity(model)

        print(f"--> Final Test Acc: {test_acc:.2f}% | Final Sparsity: {final_sparsity:.2f}%")

        # Save the best highly-pruned model for deploy.py
        if lam == 2.5:
            torch.save(model.state_dict(), "outputs/best_pruned_model.pth")
            print("Saved highly pruned model to outputs/best_pruned_model.pth")

        results.append({'lambda': lam, 'accuracy': test_acc, 'sparsity': final_sparsity})
        plot_path = f"outputs/gate_dist_lambda_{lam}.png"
        plot_gate_distributions(model, lam, save_path=plot_path)

    print_markdown_table(results)

if __name__ == "__main__":
    main()


Overwriting run_experiments.py


In [9]:
!python run_experiments.py

Executing on device: cuda:0

Starting Experiment with Lambda: 0.0
Epoch [1/20] | Loss: 1.5407 | Train Acc: 43.37% | Sparsity: 0.00%
Epoch [2/20] | Loss: 1.1617 | Train Acc: 58.32% | Sparsity: 0.00%
Epoch [3/20] | Loss: 0.9788 | Train Acc: 65.16% | Sparsity: 0.00%
Epoch [4/20] | Loss: 0.8736 | Train Acc: 69.28% | Sparsity: 0.00%
Epoch [5/20] | Loss: 0.7861 | Train Acc: 72.51% | Sparsity: 0.00%
Epoch [6/20] | Loss: 0.7405 | Train Acc: 73.93% | Sparsity: 0.00%
Epoch [7/20] | Loss: 0.6921 | Train Acc: 75.68% | Sparsity: 0.00%
Epoch [8/20] | Loss: 0.6549 | Train Acc: 76.96% | Sparsity: 0.00%
Epoch [9/20] | Loss: 0.6245 | Train Acc: 78.26% | Sparsity: 0.00%
Epoch [10/20] | Loss: 0.6022 | Train Acc: 78.99% | Sparsity: 0.00%
Epoch [11/20] | Loss: 0.5694 | Train Acc: 80.17% | Sparsity: 0.00%
Epoch [12/20] | Loss: 0.5527 | Train Acc: 80.56% | Sparsity: 0.00%
Epoch [13/20] | Loss: 0.5330 | Train Acc: 81.25% | Sparsity: 0.00%
Epoch [14/20] | Loss: 0.5182 | Train Acc: 81.98% | Sparsity: 0.00%
Epoch